In [1]:
import os
import os.path
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Local
Dataset mode: test
Query file: /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/test.csv
Validation mode: False
Force rebuild indices: False

Corpus files:
  Laws CSV: /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv (73.0 MB)
  Courts CSV: /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/court_considerations.csv (2.43 GB)

Index cache: /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed


# 2. Load Corpora and Build/Load Indices

In [3]:
from bm25index import BM25Index
import bm25index

In [4]:
# Load or build laws index
# Laws CSV: ~45MB, ~269K rows
# Build time: ~30 seconds | Load from cache: <1 second

laws_index = bm25index.get_or_build_index(
    name="laws",
    csv_path=LAWS_CSV,
    index_path=LAWS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=100000000  # Uncomment to test with smaller corpus
)
print(f"\nLaws index: {len(laws_index.documents):,} documents")

# Test search
test_results = laws_index.search("Vertrag", top_k=3)
print(f"\nTest search 'Vertrag': {len(test_results)} results")
if test_results:
    print(test_results)

Loading cached laws index from /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed/laws_index.pkl
  Loaded 175,933 documents

Laws index: 175,933 documents


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Vertrag': 1 results
[{'query': 'Vertrag', 'hits': [{'rank': 1, 'score': 2.8019, 'text': '4 Wird ein Rahmen Vertrag mit nur einer Anbieterin abgeschlossen, so Wer Den die auf Die Sem Rahmen Vertrag beruhenden Einzel Verträge entsprechend den Bedingungen des Rahmen Vertrags abgeschlossen. Für den Abschluss der Einzel Verträge kann die Auftraggeberin die jeweilige Vertrags Partnerin schriftlich auffordern, ihr Ange Bot zu vervollständigen.', 'citation': 'Art. 25 Abs. 4 BöB'}, {'rank': 2, 'score': 2.783, 'text': '6 Ist das BAKOM Registerbetreiberin, so Unter Steht der Vertrag dem öffentlichen Recht (verwaltungsrechtlicher Vertrag); ist die Auf Gabe an einen Dritten übertragen, so Unter Steht der Vertrag dem Privat Recht (privatrechtlicher Vertrag).', 'citation': 'Art. 17 Abs. 6 VID'}, {'rank': 3, 'score': 2.7276, 'text': '1 Durch Vertrag kann die Verpflichtung zum Abschluss eines künftigen Vertrages begründet werden.', 'citation': 'Art. 22 Abs. 1 OR'}]}]


In [5]:
# Load or build courts index
# Courts CSV: ~2.3GB, ~2.5M rows
# Full corpus build time: ~15-20 minutes | Load from cache: ~10 seconds
# Full corpus can have peak memory during build: ~8-16GB

courts_index = bm25index.get_or_build_index(
    name="courts",
    csv_path=COURTS_CSV,
    index_path=COURTS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=100000000  # Change to use bigger corpus
)
print(f"\nCourts index: {len(courts_index.documents):,} documents")

# Test search
test_results = courts_index.search("Meinungsfreiheit", top_k=3)
print(f"\nTest search 'Meinungsfreiheit': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")

Loading cached courts index from /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed/courts_index.pkl
  Loaded 2,476,315 documents

Courts index: 2,476,315 documents


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Meinungsfreiheit': 1 results
  Top result: N/A


In [6]:
import re
def extract_citations_from_text(text: str) -> list[str]:
    """Extract citations from any text (tool output or final answer)."""
    citations = []
    
    # SR pattern: SR followed by number (optionally with article)
    sr_matches = re.findall(
        r"SR\s*\d{3}(?:\.\d+)?(?:\s+Art\.?\s*\d+[a-z]?)?",
        text,
        re.IGNORECASE
    )
    citations.extend(sr_matches)
    
    # BGE pattern: BGE volume section page
    bge_matches = re.findall(
        r"BGE\s+\d{1,3}\s+[IVX]+[a-z]?\s+\d+(?:\s+E\.\s*\d+[a-z]?)?",
        text,
        re.IGNORECASE
    )
    citations.extend(bge_matches)
    
    # Art. pattern: Art. X LAW (e.g., Art. 1 ZGB, Art. 41 OR)
    art_matches = re.findall(
        r"Art\.?\s+\d+[a-z]?\s+(?:Abs\.?\s*\d+\s+)?[A-Z]{2,}",
        text,
        re.IGNORECASE
    )
    citations.extend(art_matches)
    
    return list(set(citations))

In [7]:
from sentence_transformers import SentenceTransformer
import faiss

model = SentenceTransformer("/root/.cache/modelscope/hub/models/BAAI/bge-m3/")


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS
/root/miniconda3/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [8]:
import query_by_dense

court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

test_df = pd.read_csv('../data/test_rewrite_001.csv')
id_l = []
citation_l = []

print("data loaded")

for id, q in zip(test_df['query_id'].tolist(), test_df['query'].tolist()):
    print("query len:", len(q))
    id_l.append(id)
    citations = []

    seen_court_citation_set = set()
    test_results = courts_index.search(q, top_k=500)
    if test_results:
        for hit in test_results[0]['hits']:
            seen_court_citation_set.add(hit['citation'])

        second_layer_test_results = []
        for hit in test_results[0]['hits']:
            for c in extract_citations_from_text(hit['text']):
                if c not in seen_court_citation_set and c in court_consideration_d:
                    seen_court_citation_set.add(c)
                    second_layer_test_results.append({'citation':c, 'text':court_consideration_d.get(c, '')})
        for second_layer_hit in second_layer_test_results:
            test_results[0]['hits'].append(second_layer_hit)

    if test_results:
        raw_hits = []
        for hit in test_results[0]['hits']:
            raw_hits.append({'citation':hit['citation'], 'text':court_consideration_d[hit['citation']]})
        court_l = query_by_dense.query_by_dense(model, q, raw_hits, 20, 1000, 200)
        # court_l = query_by_dense.query_by_dense(model, q, raw_hits, 20)
        for court in court_l:
            citations.append(court['citation'])
            for c in extract_citations_from_text(court['text']):
                citations.append(c)

    test_results = laws_index.search(q, top_k=100)
    # for i, hit in enumerate(test_results[0]['hits']):
    #     hit['citation']
    #     print("====>",i)   
    if test_results:
        raw_hits = []
        for hit in test_results[0]['hits']:
            raw_hits.append({'citation':hit['citation'], 'text':law_d[hit['citation']]})
            
        law_l = query_by_dense.query_by_dense(model, q, raw_hits, 10)
        for law in law_l:
            citations.append(law['citation'])

    citations = list(set(citations))
    citation_l.append(';'.join(citations))
    print(id)

result_df = pd.DataFrame({'query_id':id_l, 'predicted_citations':citation_l})
result_df.to_csv("../data/result.csv", index=False)

data loaded
query len: 394


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

query_by_dense_chunk_multi_vector: 100%|██████████| 1039/1039 [00:44<00:00, 23.36it/s]


...

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:01<00:00, 55.01it/s]


test_040
